# Figure 2 — A single unit as a finite-state jump process

**The process.** One exposed coordinate $\mu$ with $G$ levels on $[-1,1]$ and $d$ microstates per level, one
boundary coordinate $s$, one structural coordinate $W$ on a grid, and two presentations that alternate at rate
$1/\tau_u$. The rates fall into three classes (`network/unit.py`):
- *fast*: hops among the microstates of a level ($k_x$), and the opening and closing of encounter windows;
- *mesoscale*: encounters (rate $\kappa$), during which $\mu$ moves among all levels, and moves of the boundary;
- *slow*: structural steps $W\to W\pm\delta$, attempted at the closing of a fraction $p_W$ of windows.

Every rate obeys local detailed balance with respect to the tilted energy $(W s+v_k)\mu+q_k\mu^2$: within a
window, $\mu$ equilibrates with its current field; a structural step is accepted with the Glauber probability
$1/(1+e^{-D})$, $D=\beta\,\Delta[W s\mu-A(Ws)]+\Delta\log p_0$, evaluated at the pair $(s,\mu)$ registered at closing.
No inference or learning rule is written into the rates.

**Parameters.** Presentations $k=\pm$: boundary field $\pm6$ and an observation of $\mu$ with value $\pm0.6$ and
precision $\Pi=8$ (direct field $\pm4.8$, curvature $-4$). Rates in units of $\kappa=1$: $k_x=1000$, window closing
$k_c=100$, boundary moves $k_s=1$, $\tau_u=4$, $p_W=0.02$, evidence weight $\beta=16$, Gaussian prior on $W$ with
standard deviation 1.5.

**What is computed.** From the generator: (i) the relaxation spectrum, (ii) the stationary law of $\mu$ at fixed
field, its mean, variance and step response, (iii) the law of $W(t)$ from the blind state (Monte Carlo of the
encounter limit, and the averaged learning law) and its exact stationary law, compared with $p_0(W)\,e^{\beta\ell(W)}$.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from network.family import grid, L as L_grid, V as V_grid, langevin, cb_density
from network.unit import GatedUnit, EncounterUnit, spectrum
from network import plotting as P
P.set_style()
DATA = '../data'; os.makedirs(DATA, exist_ok=True)

PI, X, HS = 8.0, 0.6, 6.0
PRES = ((HS, PI * X, -PI / 2), (-HS, -PI * X, -PI / 2))
RATES = dict(k_x=1000.0, k_o=1000.0, k_c=100.0, kappa=1.0, k_s=1.0, tau_u=4.0)
LEARN = dict(p_W=0.02, beta=16.0, prior_sd=1.5)
GRAYF, SLOW = '#b9b8b3', P.PURPLE

## 1. Timescale separation
A small copy of the unit (7 levels, 11 structural levels) is diagonalized exactly. The rate classes were
chosen to be separated: micro hops and window closing (fast), encounters, boundary moves and presentation
switches (mesoscale), and structural steps, which occur in a fraction $p_W$ of encounters (slow).

In [2]:
fn = f'{DATA}/fig2_spectrum.npz'
if not os.path.exists(fn):
    small = GatedUnit(G=7, d=2, Gs=5, K=11, pres=PRES, **RATES, **LEARN)
    rates, ev = spectrum(small)
    np.savez(fn, rates=rates, ev=ev)
z = np.load(fn); rates = np.sort(z['rates'])
r_struct_max = rates[rates < 0.2].max(); r_meso = rates[(rates > 0.2) & (rates < 40)]; r_fast = rates[rates >= 40]
print('structure band up to', r_struct_max, '| mesoscale', r_meso.min(), '-', r_meso.max(), '| fast from', r_fast.min())
print('gaps: x%.0f (structure -> mesoscale), x%.1f (mesoscale -> fast)' % (r_meso.min() / r_struct_max, r_fast.min() / r_meso.max()))

fig, ax = P.panel(44, 40)
cls = np.where(rates < 0.2, 0, np.where(rates < 40, 1, 2))
for c_, col, nm in [(2, GRAYF, 'fast'), (1, P.BLUE, 'mesoscale'), (0, SLOW, 'structure')]:
    m = cls == c_
    ax.scatter(np.full(m.sum(), 0.0) + np.random.default_rng(c_).uniform(-0.28, 0.28, m.sum()), rates[m], s=2.2, color=col, lw=0, alpha=0.8)
    ax.text(0.42, np.exp(np.log(rates[m]).mean()), nm, color=col if c_ else SLOW, fontsize=6, va='center')
for lo, hi in [(r_struct_max, r_meso.min()), (r_meso.max(), r_fast.min())]:
    ax.annotate('', xy=(-0.5, lo), xytext=(-0.5, hi), arrowprops=dict(arrowstyle='<->', color=P.INK2, lw=0.6))
    ax.text(-0.58, np.sqrt(lo * hi), f'×{hi / lo:.0f}', fontsize=6, color=P.INK2, ha='right', va='center')
ax.set_yscale('log'); ax.set_xlim(-1.05, 1.25); ax.set_xticks([])
ax.spines['bottom'].set_visible(False); ax.set_ylabel('relaxation rate (κ)')
P.save_panel(fig, 'fig2_spectrum')

structure band up to 0.017269060299051918 | mesoscale 0.49999999999639283 - 7.212412964149133 | fast from 100.9827320370386
gaps: x29 (structure -> mesoscale), x14.0 (mesoscale -> fast)


'../fig/panels/fig2_spectrum.svg'

## 2. The continuous Bernoulli emerges from equal microstate counts
At frozen structure and boundary, the stationary law of $\mu$ under the gated kinetics is computed from the
generator for several total fields $\theta+v$. Equal numbers of microstates per level (the uniform resting law)
give $e^{(\theta+v)\mu}$ on the grid, whose continuum limit is the continuous Bernoulli density.

In [3]:
fn = f'{DATA}/fig2_cb.npz'
thetas = [-3.0, 0.0, 1.5, 4.0]; G_cb = 41
if not os.path.exists(fn):
    laws = []
    for th in thetas:
        u = GatedUnit(G=G_cb, d=2, Gs=1, K=1, pres=((0.0, th, 0.0),), **RATES, **LEARN)
        laws.append(u.marginal(u.stationary(), 'a'))
    np.savez(fn, laws=np.array(laws))
laws = np.load(fn)['laws']; mu = grid(G_cb); dmu = mu[1] - mu[0]
fig, ax = P.panel(44, 40)
xx = np.linspace(-1, 1, 400); cols4 = [P.ORANGE, P.MUTED, P.AQUA, P.BLUE]
for th, lw_, col in zip(thetas, laws, cols4):
    ax.bar(mu, lw_ / dmu, width=dmu * 0.8, color=col, alpha=0.3, lw=0)
    ax.plot(xx, cb_density(xx, th), color=col, lw=1.0)
    j = np.argmax(cb_density(xx, th)); 
for th, col, (x_, y_) in zip(thetas, cols4, [(-0.62, 2.2), (0.0, 0.72), (0.52, 1.35), (0.8, 3.3)]):
    ax.text(x_, y_, f'{th:g}', color=col, fontsize=6, ha='center')
ax.set_xlabel('exposed coordinate μ'); ax.set_ylabel('stationary density'); ax.set_xlim(-1.02, 1.02); ax.set_ylim(0, 4.2)
ax.text(0.02, 0.97, 'field θ+v', transform=ax.transAxes, fontsize=5.5, color=P.INK2, va='top')
P.save_panel(fig, 'fig2_cb')
print('max |grid law/Δμ - CB density| at cell centres:', max(np.abs(l / dmu - cb_density(mu, th)).max() for l, th in zip(laws, thetas)))

max |grid law/Δμ - CB density| at cell centres: 0.0057602749324288105


## 3. Perception
The stationary mean of $\mu$ as a function of the total field, the fluctuation–dissipation relation
(variance equals the slope of the mean response), and the relaxation of the mean after a step in the
direct field, which proceeds at the encounter rate: $\dot m=-\kappa\,(m-L(\theta+v))$.

In [4]:
fn = f'{DATA}/fig2_perception.npz'
tf = np.linspace(-6, 6, 25); G_p = 21
if not os.path.exists(fn):
    means, vars_ = [], []
    for th in tf:
        u = GatedUnit(G=G_p, d=2, Gs=1, K=1, pres=((0.0, th, 0.0),), **RATES, **LEARN)
        pa = u.marginal(u.stationary(), 'a'); mg = grid(G_p)
        means.append(pa @ mg); vars_.append(pa @ mg ** 2 - (pa @ mg) ** 2)
    # step response: stationary under v = -2, then v = +2 at t = 0
    import scipy.sparse.linalg as spla
    u0 = GatedUnit(G=G_p, d=2, Gs=1, K=1, pres=((0.0, -2.0, 0.0),), **RATES, **LEARN)
    u1 = GatedUnit(G=G_p, d=2, Gs=1, K=1, pres=((0.0, 2.0, 0.0),), **RATES, **LEARN)
    p = u0.stationary(); Q1 = u1.generator(); ts = np.linspace(0, 5, 51); mstep = []
    for i, t in enumerate(ts):
        if i: p = spla.expm_multiply(Q1.T * (ts[1] - ts[0]), p)
        mstep.append(u1.marginal(p, 'a') @ grid(G_p))
    np.savez(fn, means=np.array(means), vars=np.array(vars_), ts=ts, mstep=np.array(mstep))
z = np.load(fn); means, vars_, ts, mstep = z['means'], z['vars'], z['ts'], z['mstep']
mg = grid(G_p); slope = np.gradient(means, tf)

fig, ax = P.panel(44, 40)
tt = np.linspace(-6, 6, 300)
ax.plot(tt, langevin(tt), color=P.INK2, lw=1.0, label='Langevin')
ax.plot(tf, means, 'o', ms=2.4, color=P.BLUE, label='jump process', mew=0)
ax.set_xlabel('field θ + v'); ax.set_ylabel('mean of μ'); ax.set_yticks([-1, 0, 1]); ax.set_ylim(-1.05, 1.05)
ax.legend(loc='upper left', fontsize=5.5, handlelength=1.2)
ins = ax.inset_axes([0.64, 0.12, 0.33, 0.36])
ins.plot([0, 0.35], [0, 0.35], color=P.INK2, lw=0.7)
ins.plot(vars_, slope, 'o', ms=1.6, color=P.BLUE, mew=0)
ins.set_xlabel('variance', fontsize=5, labelpad=0.5); ins.set_ylabel('gain', fontsize=5, labelpad=0.5)
ins.tick_params(labelsize=4.5, length=1.5, pad=1); ins.set_xticks([0, 0.3]); ins.set_yticks([0, 0.3])
P.save_panel(fig, 'fig2_meanmap')
print('max |mean - L_grid|:', np.abs(means - L_grid(tf, mg)).max(), ' max |mean - Langevin|:', np.abs(means - langevin(tf)).max(),
      ' FDT max |slope - var| (interior):', np.abs(slope - vars_)[2:-2].max())

Lold, Lnew = L_grid(-2.0, mg), L_grid(2.0, mg)
print('max |step response - exponential|:', np.abs(mstep - (Lnew + (Lold - Lnew) * np.exp(-ts))).max())

max |mean - L_grid|: 6.761258219967203e-14  max |mean - Langevin|: 0.004510656648420541  FDT max |slope - var| (interior): 0.0054264772914901505
max |step response - exponential|: 0.020640167733740306


## 4. Learning
The same unit with a slow structural coordinate. The environment alternates two presentations in which the
boundary field and the observed value of $\mu$ agree in sign. Sample paths of $W(t)$ start from the blind state
($W=0$; Monte Carlo of the encounter limit, i.e. complete equilibration within windows). They are compared with
the learning law: the structural chain whose step rates are averaged over the pairs $(s,\mu')$ registered at
encounters. The stationary law is computed exactly and compared with $p_0(W)\,e^{\beta\ell(W)}$, where $\ell$ is
the average log-evidence. A check at coarser resolution confirms that the full gated chain and its encounter
limit have the same structural law.

In [5]:
from network.sim import unit_mc
import scipy.linalg as sla
fn = f'{DATA}/fig2_learning.npz'
enc_kw = {k: RATES[k] for k in ('kappa', 'k_s', 'tau_u')}
T_L = 100000.0
if not os.path.exists(fn):
    e = EncounterUnit(G=11, Gs=5, K=201, Wmax=5.0, pres=PRES, **enc_kw, **LEARN)
    t_mc, W_mc = unit_mc(e, T_L, n_paths=64, n_rec=200, seed=1)
    Qw = e.averaged_W_generator(); p0 = np.zeros(len(e.W)); p0[len(e.W) // 2] = 1.0
    t_th = np.linspace(0, T_L, 101); law_th = np.array([p0 @ sla.expm(Qw * t) for t in t_th])
    W_enc = e.W_marginal(e.stationary()); W_th = e.theory_W_law()
    # gated chain vs its encounter limit, coarser structural grid
    g_c = GatedUnit(G=11, d=1, Gs=5, K=121, Wmax=4.0, pres=PRES, **RATES, **LEARN)
    e_c = EncounterUnit(G=11, Gs=5, K=121, Wmax=4.0, pres=PRES, **enc_kw, **LEARN)
    Wg_c = g_c.marginal(g_c.stationary(), 'c'); We_c = e_c.W_marginal(e_c.stationary())
    np.savez(fn, t_mc=t_mc, W_mc=W_mc, t_th=t_th, law_th=law_th, W_enc=W_enc, W_th=W_th, Wgrid=e.W,
             Wg_c=Wg_c, We_c=We_c, Wgrid_c=e_c.W)
z = np.load(fn); Wg = z['Wgrid']
tv = lambda a, b: 0.5 * np.abs(a - b).sum()
late = z['W_mc'][:, z['W_mc'].shape[1] // 2:].ravel()
eps = Wg[1] - Wg[0]
hist = np.histogram(late, bins=np.append(Wg - eps / 2, Wg[-1] + eps / 2))[0] / late.size
res = {'TV_exact_vs_theory': tv(z['W_enc'], z['W_th']), 'TV_gated_vs_encounter_K121': tv(z['Wg_c'], z['We_c']),
       'mean_W_exact': float(z['W_enc'] @ Wg), 'mean_W_theory': float(z['W_th'] @ Wg),
       'sd_W_theory': float(np.sqrt(z['W_th'] @ Wg ** 2 - (z['W_th'] @ Wg) ** 2)),
       'mean_W_MC_late': float(late.mean()), 'mean_W_MC_final': float(z['W_mc'][:, -1].mean()),
       'mean_W_learning_law_final': float(z['law_th'][-1] @ Wg)}
print(res)

fig = plt.figure(figsize=(44 * P.MM, 40 * P.MM), layout='constrained')
gs = fig.add_gridspec(1, 2, width_ratios=[3.2, 1], wspace=0.02)
ax = fig.add_subplot(gs[0]); axr = fig.add_subplot(gs[1], sharey=ax)
cdf = np.cumsum(z['law_th'], 1); q = lambda f: Wg[np.argmax(cdf >= f, 1)]
ax.fill_between(z['t_th'] / 1e4, q(0.1), q(0.9), color=P.MUTED, alpha=0.2, lw=0)
for p_ in z['W_mc'][:4]:
    ax.plot(z['t_mc'] / 1e4, p_, color=SLOW, lw=0.35, alpha=0.55)
ax.plot(z['t_mc'] / 1e4, z['W_mc'].mean(0), color=SLOW, lw=1.3)
ax.plot(z['t_th'] / 1e4, z['law_th'] @ Wg, '--', color=P.INK, lw=0.9)
ax.axhline(0, color=P.GRID, lw=0.6, zorder=0)
ax.set_xlabel('time (10⁴/κ)'); ax.set_ylabel('structure W'); ax.set_xlim(0, T_L / 1e4); ax.set_ylim(-1.0, 4.5)
axr.fill_betweenx(Wg, 0, z['W_enc'] / eps, color=SLOW, alpha=0.35, lw=0)
axr.plot(z['W_th'] / eps, Wg, color=P.INK, lw=0.9)
axr.set_xticks([]); axr.tick_params(labelleft=False); axr.spines['bottom'].set_visible(False); axr.set_xlim(0, None)
P.save_panel(fig, 'fig2_learning')
json.dump({k: float(v) for k, v in res.items()}, open(f'{DATA}/fig2_results.json', 'w'), indent=1)

{'TV_exact_vs_theory': np.float64(0.007737673272832847), 'TV_gated_vs_encounter_K121': np.float64(0.006627781368889626), 'mean_W_exact': 2.178638909690837, 'mean_W_theory': 2.164102822854569, 'sd_W_theory': 0.7966660845349051, 'mean_W_MC_late': 1.6534609375000002, 'mean_W_MC_final': 1.8750000000000004, 'mean_W_learning_law_final': 1.926946897170025}


## Composite

In [6]:
pp = '../fig/panels/'
P.compose('../fig/fig2_single_unit.svg', 180, 44,
          panels=[(pp + 'fig2_spectrum.svg', 0, 3), (pp + 'fig2_cb.svg', 45, 3), (pp + 'fig2_meanmap.svg', 90, 3), (pp + 'fig2_learning.svg', 135, 3)],
          labels=[('A', 1.5, 4.5), ('B', 46.5, 4.5), ('C', 91.5, 4.5), ('D', 136.5, 4.5)])

'../fig/fig2_single_unit.svg'